In [0]:
from pyspark.sql.functions import col, lower, when, dense_rank
from pyspark.sql.window import Window

# Reading in bronze Delta table (raw data)

df = spark.read.table("marathos.bronze.races")


## Filtering Invalid Rows
Removing rows per: d unit events, null values, and 69 invalid These 69 rows are very few to affect analysis and represent data quality issues.

In [0]:
#   Remove invalid row:
# - d unit days = explicitly invalid per instructions
# - null and the None values removed
# - 69 invalid distance events removed (km/mi without h performance
# Decision: these rows are too few (69) to affect analysis and are data quality issues
df_filtered = df.filter(
    ~col("event_distance_length").endswith("d") &
    col("event_distance_length").isNotNull() &
    (col("event_distance_length") != "None")
).filter(
    ~(
        (
            lower(col("event_distance_length")).contains("km") |
            lower(col("event_distance_length")).contains("mi")
        ) &
        ~col("athlete_performance").contains("h")
    )
)

print(f"Rows before: {df.count()}")
print(f"Rows after: {df_filtered.count()}")
print(f"Rows removed: {df.count() - df_filtered.count()}")

 ## Validity check
 
 Here i start verifing the validity of how the distance and and time is set.
    - this is important beacuse it gives us insight on how the dataset is set up. And if there is a diffrence then we get to to take it in to acount.


In [0]:
# Verify data validity per instructions:
# km/mi events must have performance in h
# h events must have performance in km
invalid_distance = df_filtered.filter(
    (
        lower(col("event_distance_length")).contains("km") |
        lower(col("event_distance_length")).contains("mi")
    ) &
    ~col("athlete_performance").contains("h")
)

invalid_time = df_filtered.filter(
    lower(col("event_distance_length")).endswith("h") &
    ~col("athlete_performance").contains("km")
)

print(f"Invalid distance events: {invalid_distance.count()}")
print(f"Invalid time events: {invalid_time.count()}")

## Performance Conversion
Converting athlete_performance to number types. Distance event (km/mi) converted to seconds. Time events (h) converted to km. Multi-day format (2d+) handled by UDF. 

Here i made a deicition to have a sanity check on all speed that was over 25 (h) which i do not find realistic. 

In [0]:
from pyspark.sql.functions import udf, regexp_extract # Source: LLM assisted with with finding these imports UDF and regexp_extract 
from pyspark.sql.types import DoubleType
from pyspark.sql.functions import expr

# UDF handles two times formats in data:
# 12:33:00 h = normal format
# "2d 00:56:24 h" = multi-day format (valid race results over 24h)
def parse_performance(par_per):
    if par_per is None:
        return None
    try:
        par_per = par_per.strip()
        if 'd' in par_per and ':' in par_per:
            parts = par_per.split('d')
            days = int(parts[0].strip())
            time_part = parts[1].strip().replace(' h', '').strip()
            h, m, s = time_part.split(':')
            return float(days * 86400 + int(h) * 3600 + int(m) * 60 + float(s))
        elif ':' in par_per:
            time_part = par_per.replace(' h', '').strip()
            h, m, s = time_part.split(':')
            return float(int(h) * 3600 + int(m) * 60 + float(s))
        else:
            return None
    except:
        return None

parse_udf = udf(parse_performance, DoubleType())

df_converted = df_filtered.withColumn(
    "performance_seconds",
    when(
        lower(col("event_distance_length")).contains("km") |
        lower(col("event_distance_length")).contains("mi"),
        parse_udf(col("athlete_performance"))
    ).otherwise(None)
).withColumn(
    "performance_km",
    when(
        lower(col("event_distance_length")).endswith("h"),
        regexp_extract(col("athlete_performance"), r"([\d.]+)", 1).cast("double")
    ).otherwise(None)
)

print(f"Rows with performance_seconds: {df_converted.filter(col('performance_seconds').isNotNull()).count()}")
print(f"Rows with performance_km: {df_converted.filter(col('performance_km').isNotNull()).count()}")

# Here i am adding a sanity check on athelete_average_speed, due to the conversation with the stakeholder.

# speed that is > 25h is not realistic in a ultra marathon - i will be removing to keep the data in good quality.

df_converted = df_converted.withColumn(
    "athlete_average_speed",
    expr("try_cast(athlete_average_speed as double)")
).filter(
    col("athlete_average_speed").isNull() |
    (col("athlete_average_speed") <= 25.0)
)
print(f"rows after speed sanity check: {df_converted.count()}")




## **Creating IDs and Handeling Nulls**

Here i am creating event_id and result_id using sha2 as recommended by stakeholder.


In [0]:
from pyspark.sql.functions import sha2, concat_ws 
# Creating event_id using sha2 on the event_name

df_with_ids = df_converted.withColumn(
    "event_id", sha2(col("event_name"), 256)
)
#creating result_id as the primary key ( unique per athlete + event + year)

df_with_ids = df_with_ids.withColumn(
    "result_id", sha2(
        concat_ws("||",
            col("athlete_id").cast("string"),
            col("event_name"),
            col("year_of_event").cast("string")
        ), 256
    )
)

# handling the nulls and cast data types
df_with_ids = df_with_ids.fillna("Missing", subset=[
    "athlete_country",
    "athlete_gender", 
    "athlete_age_category",
    "athlete_club"
]).withColumn(
    "athlete_year_of_birth",
    expr("try_cast(athlete_year_of_birth as int)")
)
print(f"Unique event_ids: {df_with_ids.select('event_id').distinct().count()}")
print(f"Unique result_ids: {df_with_ids.select('result_id').distinct().count()}")
print(f"Total rows: {df_with_ids.count()}")

# Writing to --> silver
Saving cleaned OBT to marathos.silver_races_obt. 
So then the Gold layer can read from this table.

In [0]:
# Write cleaned One Big Table to silver layer.
df_with_ids.write.format("delta").saveAsTable("marathos.silver.races_obt")
print("Silver OBT has been loaded succsesfully")